In [1]:
# ============================================================
# Supplementary Table S1
# Curated R-loop regulator gene list used in this study
# ============================================================

import os
import re
import pandas as pd

# =========================
# 1. 路径设置
# =========================

PROJECT_DIR = "/mnt/zhangzheng_group/liuz-53/Test_R/R_loop_HCC"
DATA_DIR = os.path.join(PROJECT_DIR, "00_datacollection")
OUT_DIR = os.path.join(PROJECT_DIR, "00_datacollection")

os.makedirs(OUT_DIR, exist_ok=True)

RLOOP_FILE = os.path.join(DATA_DIR, "gene_list_of_R-loop_regulators.txt")

OUT_FILE = os.path.join(
    OUT_DIR,
    "SupplementaryTable_S1_curated_Rloop_regulators_used_in_this_study.csv"
)

OUT_EXCEL = os.path.join(
    OUT_DIR,
    "SupplementaryTable_S1_curated_Rloop_regulators_used_in_this_study.xlsx"
)

# =========================
# 2. 读取 R-loop regulator list
# =========================

df = pd.read_csv(RLOOP_FILE, sep="\t")

print("Original columns:")
print(df.columns.tolist())
print("Original shape:", df.shape)

# 去除空列
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# =========================
# 3. 标准化列名
# =========================

rename_dict = {
    "Regulator": "Gene Symbol",
    "Description": "Gene name",
    "Tier": "Evidence tier",
    "Evidence (PMID)": "Evidence summary",
    "References": "Reference PMIDs",
    "Function (PMID)": "Functional evidence"
}

df = df.rename(columns=rename_dict)

# =========================
# 4. 只保留人类基因
# =========================

if "Species" in df.columns:
    df = df[df["Species"] == "Homo sapiens"].copy()

# =========================
# 5. 提取代表性 PMID
# =========================

def extract_first_pmid(x):
    if pd.isna(x):
        return ""
    pmids = re.findall(r"\b\d{7,8}\b", str(x))
    if len(pmids) == 0:
        return ""
    return pmids[0]

df["Representative PMID"] = df["Reference PMIDs"].apply(extract_first_pmid)

# =========================
# 6. 根据描述和功能自动粗略分类
# =========================

def infer_category(row):
    text = " ".join([
        str(row.get("Gene Symbol", "")),
        str(row.get("Gene name", "")),
        str(row.get("Functional evidence", "")),
        str(row.get("Evidence summary", ""))
    ]).lower()

    if "rnase" in text or "ribonuclease h" in text:
        return "RNase H / nuclease"
    elif "helicase" in text or row.get("Gene Symbol", "").startswith(("DDX", "DHX")):
        return "RNA/DNA helicase"
    elif "topoisomerase" in text or row.get("Gene Symbol", "").startswith("TOP"):
        return "Topoisomerase"
    elif "splicing" in text or "spliceosome" in text or "srsf" in text:
        return "RNA splicing / processing"
    elif "dna repair" in text or "brca" in text or "fanconi" in text or "rad" in text:
        return "DNA repair / genome stability"
    elif "chromatin" in text or "histone" in text or "methyltransferase" in text:
        return "Chromatin regulation"
    elif "transcription" in text or "rna polymerase" in text:
        return "Transcription regulation"
    elif "mitochondrial" in text or "mitochondria" in text:
        return "Mitochondrial R-loop regulation"
    elif "immune" in text or "interferon" in text or "cytosolic" in text:
        return "Immune / cytosolic RNA:DNA hybrid sensing"
    else:
        return "Other R-loop-associated regulator"

df["Functional category"] = df.apply(infer_category, axis=1)

# =========================
# 7. 标记是否进入后续分析
# =========================
# 如果你后续已经有 NMF loading 文件，可以自动判断该基因是否进入 NMF
# 如果文件不存在，则全部标记为 Not assessed

NMF_WEIGHT_FILE = os.path.join(
    PROJECT_DIR,
    "02_Figure2_python",
    "Fig2_NMF_Rloop_program_gene_weights.csv"
)

if os.path.exists(NMF_WEIGHT_FILE):
    H_df = pd.read_csv(NMF_WEIGHT_FILE, index_col=0)
    nmf_genes = set(H_df.columns.astype(str))
    df["Included in NMF analysis"] = df["Gene Symbol"].astype(str).apply(
        lambda x: "Yes" if x in nmf_genes else "No"
    )
else:
    df["Included in NMF analysis"] = "Not assessed"

# =========================
# 8. 整理投稿版表格
# =========================

keep_cols = [
    "Gene Symbol",
    "Gene name",
    "Functional category",
    "Evidence tier",
    "Included in NMF analysis",
    "Representative PMID",
    "Reference PMIDs"
]

supp_table = df[keep_cols].copy()

supp_table = supp_table.drop_duplicates(subset=["Gene Symbol"])

supp_table = supp_table.sort_values(
    by=["Evidence tier", "Gene Symbol"],
    ascending=[True, True]
)

# =========================
# 9. 保存
# =========================

supp_table.to_csv(OUT_FILE, index=False)
supp_table.to_excel(OUT_EXCEL, index=False)

print("Supplementary Table S1 saved:")
print(OUT_FILE)
print(OUT_EXCEL)

print("\nTable shape:", supp_table.shape)
print("\nEvidence tier counts:")
print(supp_table["Evidence tier"].value_counts(dropna=False))

print("\nIncluded in NMF analysis:")
print(supp_table["Included in NMF analysis"].value_counts(dropna=False))

print("\nPreview:")
print(supp_table.head())

Original columns:
['Species', 'Entrez ID', 'Regulator', 'Description', 'Evidence (PMID)', 'Tier', 'References', 'Function (PMID)', 'Unnamed: 8']
Original shape: (1293, 9)
Supplementary Table S1 saved:
/mnt/zhangzheng_group/liuz-53/Test_R/R_loop_HCC/00_datacollection/SupplementaryTable_S1_curated_Rloop_regulators_used_in_this_study.csv
/mnt/zhangzheng_group/liuz-53/Test_R/R_loop_HCC/00_datacollection/SupplementaryTable_S1_curated_Rloop_regulators_used_in_this_study.xlsx

Table shape: (1185, 7)

Evidence tier counts:
Evidence tier
5    820
4    179
3     69
2     64
1     53
Name: count, dtype: int64

Included in NMF analysis:
Included in NMF analysis
Yes    1095
No       90
Name: count, dtype: int64

Preview:
  Gene Symbol                                         Gene name  \
0        ADAR  Double-stranded RNA-specific adenosine deaminase   
1         AQR                   Intron-binding protein aquarius   
2         ATM                         Serine-protein kinase ATM   
3         ATR 